# Polynomial Regression — learn curvature without losing honest evaluation
Real problem → visual intuition → experiment → mathematics → Python → evaluation → comparison.
A straight line can miss curved housing patterns. We transform the inputs, not the meaning of linear coefficient weighting. Four lessons cover feature maps, interactions/capacity, fold-local model selection and limits. These fictional tiny datasets are not deployment evidence or a capstone.
Trusted author code only. No credentials, uploads, paid services, remote data or installation cells. Use the exact approved local profile; genuine Google Colab execution remains unverified.


In [ ]:
import sys, json, random
from importlib.metadata import version
assert sys.version_info[:3] == (3, 13, 16), 'Use the approved local Python profile, or separately verify a new profile.'
pins = {'numpy': '2.5.4', 'pandas': '3.0.6', 'matplotlib': '3.11.2', 'scikit-learn': '1.9.1'}
for package, expected in pins.items():
    assert version(package) == expected, f'Unsupported {package} profile'
random.seed(0)
print('Approved local profile asserted; seed=0')

## PR01 / PR02 — predict the columns before syntax
For dimensionless normalized area x=2, φ=[x,x²]=[2,4]. With w=[2,3], b=1 thousand USD, prediction17. For [a,b]=[2,3], degree2 produces [2,3,4,6,9]. Columns are not additional houses.
Predict: if the first weight increases by2 with input fixed, does the coefficient relationship become nonlinear? What if two input columns are swapped?


In [ ]:
import numpy as np
import pandas as pd
import math, hashlib
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error
np.random.seed(1729)
one = PolynomialFeatures(2, include_bias=False).fit_transform([[2]])[0]
two = PolynomialFeatures(2, include_bias=False).fit_transform([[2,3]])[0]
assert one.tolist() == [2,4]
assert two.tolist() == [2,3,4,6,9]
assert float(one @ np.array([2,3]) + 1) == 17
assert float(one @ np.array([4,3]) + 1) == 21
assert math.comb(2+3,3)-1 == 9
unsupported_columns = math.comb(2+8,8)-1
assert unsupported_columns == 44 and unsupported_columns > 32
print(json.dumps({"degree2_features":two.tolist(),"prediction":17,"changed_weight_prediction":21,"unsupported_columns":unsupported_columns},sort_keys=True))


## PR03 — same rows, different capacity
Predict which train and development errors will change as degree increases. A fixed map may transform validation inputs, but learned scaling must fit only training rows. This bounded Ridge(alpha1) comparison is part of the polynomial workflow, not the later regularization course.
Training and development values below match PR-E2. Final rows are separated and must not guide degree, penalty or scaling choices. A seed ensures repeatability, not uncertainty removal.


In [ ]:
TRAIN = {"ids":["h1","h2","h3","h4","h5"],"x":[-1,-.5,0,.5,1],"y":[24,18.5,21,23.5,31]}
DEV = {"ids":["h6","h7"],"x":[-.75,.75],"y":[20.75,28.25]}
TEST = {"ids":["h8","h9"],"x":[-.25,.25],"y":[19.25,21.75]}
dataset = {"train":TRAIN,"development":DEV,"final_test":TEST}
dataset_sha256 = hashlib.sha256(json.dumps(dataset,sort_keys=True,separators=(",",":")).encode()).hexdigest()
assert len(set(TRAIN["ids"]+DEV["ids"]+TEST["ids"])) == 9
X, y = np.array(TRAIN["x"]).reshape(-1,1), np.array(TRAIN["y"])
X_dev, y_dev = np.array(DEV["x"]).reshape(-1,1), np.array(DEV["y"])
def workflow(degree, penalty):
    estimator = LinearRegression() if penalty == "none" else Ridge(alpha=1,solver="svd")
    return make_pipeline(PolynomialFeatures(degree,include_bias=False),StandardScaler(),estimator)
comparison=[]
for degree in [1,3,9]:
    fitted_model=workflow(degree,"none").fit(X,y)
    comparison.append({"degree":degree,"train_mae":float(mean_absolute_error(y,fitted_model.predict(X))),
       "development_mae":float(mean_absolute_error(y_dev,fitted_model.predict(X_dev)))})
assert comparison[2]["train_mae"] < comparison[1]["train_mae"]
assert comparison[2]["development_mae"] > comparison[1]["development_mae"]
print(json.dumps({"comparison":comparison,"data_sha256":dataset_sha256},sort_keys=True))


### Safe selection and fold receipts
Choose inside cross-validation on the training/development resource, not final test. Each clone of the pipeline learns its scaler on its own fold-training IDs. After the choice is frozen, the final evaluation can use a refit on all permitted development rows. Degree9 with these tiny samples is underdetermined: do not interpret its coefficients as stable unique effects.


In [ ]:
folds=list(KFold(n_splits=3,shuffle=True,random_state=1729).split(X))
receipts=[]
for train_indices,validation_indices in folds:
    fold_train_ids=[TRAIN["ids"][index] for index in train_indices]
    fold_validation_ids=[TRAIN["ids"][index] for index in validation_indices]
    assert not set(fold_train_ids)&set(fold_validation_ids)
    local=workflow(3,"none").fit(X[train_indices],y[train_indices])
    manual_columns=PolynomialFeatures(3,include_bias=False).fit_transform(X[train_indices])
    assert np.allclose(local[1].mean_,manual_columns.mean(axis=0))
    receipts.append({"fit_ids":fold_train_ids,"validation_ids":fold_validation_ids})
candidates=[]
for degree in [1,3,9]:
    for penalty in ["none","ridge1"]:
        errors=-cross_val_score(workflow(degree,penalty),X,y,cv=folds,scoring="neg_mean_absolute_error")
        candidates.append({"degree":degree,"penalty":penalty,"cv_mae":float(errors.mean())})
selected=min(candidates,key=lambda item:(item["cv_mae"],item["degree"],item["penalty"]))
model=workflow(selected["degree"],selected["penalty"]).fit(X,y)
development_mae=float(mean_absolute_error(y_dev,model.predict(X_dev)))
selection_frozen=True
assert selection_frozen
X_refit=np.concatenate([X,X_dev])
y_refit=np.concatenate([y,y_dev])
model.fit(X_refit,y_refit)
final_predictions=model.predict(np.array(TEST["x"]).reshape(-1,1))
final_mae=float(mean_absolute_error(TEST["y"],final_predictions))
print(json.dumps({"candidates":candidates,"selected":selected,"fold_receipts":receipts,
 "development_mae_after_selection":development_mae,"final_mae":final_mae},sort_keys=True))


## PR04 — diagnose, compare and report limitations
Predict: does an accurate interpolation imply a safe tail at x=10? Can a house-price regressor be used unchanged for churn or Good/Needs Improvement labels? Neither conclusion follows.
For f(x)=x², values4 at2 and100 at10 illustrate mathematical tail growth, not safety. In multiple features, marginal ranges alone do not establish joint support. A supplied piecewise rule illustrates different assumptions; it is not a fitted tree estimator.


In [ ]:
safe_order = ["poly","scale","model"]
repair = ["poly","scale","model"]
assert repair == safe_order, "Use fixed expansion, fold-local learned scaling, then the estimator."
query=np.array([[.4],[1.5]])
predicted=model.predict(query)
support=["in-range, not a reliability guarantee","outside original training range"]
piecewise_illustration=np.where(query.ravel()<=0,20.,30.)
assert 2**2 == 4 and 10**2 == 100
report={"target":"price in thousand USD","schema":"fixed polynomial terms in sklearn order; include_bias=False; one intercept",
 "selection":selected,"selection_source":"training CV, not final test",
 "fit_ids":TRAIN["ids"]+DEV["ids"],"test_ids":TEST["ids"],"seed":1729,
 "limitations":["tiny fictional sample","rank-deficient high degrees","unsupported tails","not a classification probability","no causal guarantee"],
 "colab":"UNVERIFIED: local kernels only"}
print(json.dumps({"query_predictions":predicted.tolist(),"support":support,
 "piecewise_rule_not_fitted_tree":piecewise_illustration.tolist(),"report":report},sort_keys=True))
print(json.dumps({"rows":9,"candidates":len(candidates),"folds":len(folds),
 "feature_row":two.tolist(),"data_sha256":dataset_sha256,"mastery":"not awarded by notebook"},sort_keys=True))


## Independent transfer
1. Explain why squared inputs do not square the fitted coefficients.
2. Count columns before requesting a high-degree design and name the fitted schema.
3. Repair globally fitted scaling before CV; distinguish a fixed formula from learned statistics.
4. Compare train/development residuals on identical rows, then state why a tail remains unsupported.
Complete a fresh full platform independent form:13/16 plus both atoms of all four gates. Practice and notebook execution never award module mastery.
